# Çözücüler ⚙️

Bu egzersizde, farklı `çözücülerin` `LogisticRegression` modelleri üzerindeki etkilerini araştıracaksınız.

👇 Veri kümesini içe aktarmak için aşağıdaki kodu çalıştırın

In [1]:
import pandas as pd

df = pd.read_csv("https://d32aokrjazspmn.cloudfront.net/materials/solvers_dataset.csv")
df.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,sulphates,alcohol,quality rating
0,9.47,5.97,7.36,10.17,6.84,9.15,9.78,9.52,10.34,8.80,6
1,10.05,8.84,9.76,8.38,10.15,6.91,9.70,9.01,9.23,8.80,7
2,10.59,10.71,10.84,10.97,9.03,10.42,11.46,11.25,11.34,9.06,4
3,11.00,8.44,8.32,9.65,7.87,10.92,6.97,11.07,10.66,8.89,8
4,12.12,13.44,10.35,9.95,11.09,9.38,10.22,9.04,7.68,11.38,3


- Veri kümesi farklı şaraplardan oluşmaktadır 🍷
- Özellikler şarapların farklı niteliklerini tanımlar 
- Hedef 🎯 bir uzman tarafından verilen kalite değerlendirmesidir

## 1. Hedef mühendisliği

Bu bölümde, değerlendirmeleri ikili bir hedefe dönüştüreceksiniz.

👇 Her değerlendirme için kaç gözlem bulunmaktadır?

In [2]:
df['quality rating'].value_counts().sort_index()

quality rating
1     10090
2     10030
3      9838
4      9928
5     10124
6      9961
7      9954
8      9977
9      9955
10    10143
Name: count, dtype: int64

❓ Hedefi ikili sınıflandırma görevine dönüştürerek `y` oluşturun, burada 6'nın altındaki kalite değerlendirmeleri kötü [0], 6 ve üzeri değerlendirmeler iyi [1] olacak

In [3]:
y = (df['quality rating'] >= 6).astype(int)
y.value_counts()

quality rating
0    50010
1    49990
Name: count, dtype: int64

❓ Yeni ikili hedefin sınıf dengesini kontrol edin

In [4]:
from sklearn.preprocessing import StandardScaler

X = df.drop(columns=['quality rating'])

scaler = StandardScaler()
X = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)
X.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,sulphates,alcohol
0,-0.788603,-1.528461,-1.733180,0.461130,-1.526653,-0.852381,-0.221393,-0.478387,0.340231,-0.489833
1,-0.346860,-0.462069,-0.158290,-0.783868,0.117066,-3.102634,-0.301357,-0.986972,-0.769429,-0.489833
2,0.064417,0.232757,0.550411,1.017553,-0.439117,0.423432,1.457850,1.246811,1.339925,-0.307387
3,0.376684,-0.610695,-1.103224,0.099454,-1.015163,0.925720,-3.030126,1.067310,0.660133,-0.426679
4,1.229704,1.247129,0.228871,0.308113,0.583862,-0.621328,0.218409,-0.957055,-2.318954,1.320591


❓ Özellikleri normalleştirerek `X`'inizi oluşturun. Bu farklı çözücülerin adil karşılaştırılmasına olanak sağlayacaktır.

In [6]:
# YOUR CODE HERE

In [ ]:
# YOUR CODE HERE

## 2. LogisticRegression çözücüleri

❓ Lojistik Regresyon modelleri farklı **çözücüler** kullanılarak optimize edilebilir. Mevcut çözücülerin karşılaştırmasını yapın:
- Uyum süresi - hangi çözücü **en hızlı**?
- Kesinlik - kesinlik puanları **ne kadar farklı**?

Lojistik Regresyon için mevcut çözücüler: `['newton-cg', 'lbfgs', 'liblinear', 'sag', 'saga']`
 
Bu 5 çözücü hakkında daha fazla bilgi için [bu Stack Overflow konusuna](https://stackoverflow.com/questions/38640109/logistic-regression-python-solvers-defintions) göz atın

In [7]:
from sklearn.linear_model import LogisticRegression
from time import time

solvers = ['newton-cg', 'lbfgs', 'liblinear', 'sag', 'saga']
results = {}

for solver in solvers:
    model = LogisticRegression(solver=solver, max_iter=1000)
    start = time()
    model.fit(X, y)
    elapsed = time() - start
    accuracy = model.score(X, y)
    results[solver] = {'time': elapsed, 'accuracy': accuracy}
    print(f"{solver}: time={elapsed:.3f}s, accuracy={accuracy:.4f}")

newton-cg: time=0.187s, accuracy=0.8611
lbfgs: time=0.122s, accuracy=0.8611
liblinear: time=0.161s, accuracy=0.8612
sag: time=0.827s, accuracy=0.8612
saga: time=1.630s, accuracy=0.8612


In [8]:
fastest_solver = min(results, key=lambda s: results[s]['time'])
fastest_solver

'lbfgs'

<details>
    <summary>ℹ️ Yorumumuz için buraya tıklayın</summary>

Maliyet fonksiyonumuz 5 çözücünün de bulduğu global bir minimuma sahip olacak kadar "kolay" olduğundan, tüm çözücüler benzer kesinlik puanları üretmelidir. Derin Öğrenme'de olduğu gibi çok karmaşık maliyet fonksiyonları için, farklı çözücüler kayıp fonksiyonunun farklı değerlerinde durabilir.

**Şarap veri kümesi**
    
Mevcut veri kümesinde sklearn'in <a href="https://scikit-learn.org/stable/modules/generated/sklearn.inspection.permutation_importance.html">permutation_importance</a> ile özellik önemini kontrol ederseniz, birçok özelliğin neredeyse 0 önemine sahip olduğunu göreceksiniz. Liblinear çözücü, bir defada sadece *bir* yön boyunca hareket eder ve diğerlerini L1 düzenlileştirme ile düzenler (yani, beta değerlerini 0'a ayarlar), bu da birçok özelliğin hedefi tahmin etmede o kadar da önemli olmadığı bir veri kümesi için iyi bir uyum sağlayabilir.

❗️En iyi çözücüyü arama maliyeti vardır. Varsayılanla (`lbfgs`) devam etmek genel olarak en çok zaman tasarrufu sağlayabilir, sklearn başlamak için hangi çözücüyü seçeceğiniz konusunda fikir vermek için bu tabloyu sunar: 

<img src="https://wagon-public-datasets.s3.amazonaws.com/05-Machine-Learning/04-Under-the-Hood/solvers-chart.png" width=700>

</details>

###  🧪 Kodunuzu test edin

In [9]:
from nbresult import ChallengeResult

result = ChallengeResult(
    'solvers',
    fastest_solver=fastest_solver
)
result.write()
print(result.check())


============================= test session starts ==============================
platform linux -- Python 3.12.9, pytest-9.1.1, pluggy-1.6.0 -- /home/kaan/.pyenv/versions/3.12.9/envs/workintech/bin/python
cachedir: .pytest_cache
rootdir: /home/kaan/code/sprint_215/S16D4-S-data-solvers/tests
plugins: anyio-4.8.0, typeguard-4.4.2, dash-4.4.1
collecting ... collected 1 item

test_solvers.py::TestSolvers::test_fastest_solver PASSED                 [100%]

============================== 1 passed in 0.01s ===============================


💯 You can commit your code:

git add tests/solvers.pickle

git commit -m 'Completed solvers step'

git push origin master



## 3. Stokastik Gradyan İnişi

Lojistik Regresyon modelleri ayrıca Stokastik Gradyan İnişi ile de optimize edilebilir.

❓ **Stokastik Gradyan İnişi** ile optimize edilmiş bir Lojistik Regresyon modelini değerlendirin. Kesinlik puanı ve eğitim süresi 2. bölümde eğitilen modellerin performansı ile nasıl karşılaştırılır?

<details>
<summary>💡 İpucu</summary>

- Takılırsanız, [SGDClassifier belgelerine](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.SGDClassifier.html) bakın!

</details>

In [10]:
from sklearn.linear_model import SGDClassifier

sgd = SGDClassifier(loss='log_loss', max_iter=1000)

start = time()
sgd.fit(X, y)
sgd_time = time() - start
sgd_accuracy = sgd.score(X, y)

print(f"SGD: time={sgd_time:.3f}s, accuracy={sgd_accuracy:.4f}")

SGD: time=0.133s, accuracy=0.8610


☝️ SGD modeli, benzer performans için en kısa sürelerden birine sahip olmalıdır (hatta `liblinear`'dan bile daha kısa olabilir). Bu, Gradyan İnişinin her dönemini aynı anda 100k satırı belleğe yüklemek yerine tek bir satırda gerçekleştirmenin doğrudan bir etkisidir.

## 4. Tahminler

❓ En iyi modeli (kısa uyum süresi ve yüksek kesinlik ile dengelenen) kullanarak aşağıdaki şarabın ikili kalitesini (0 veya 1) tahmin edin. Şunları kaydedin:
- `predicted_class`
- `predicted_proba_of_class` (yani modeliniz 1 sınıfını tahmin ettiyse, 1'in sınıf olması gerektiğine inanma olasılığı nedir, 0 ile 1 arasında olmalıdır)

In [11]:
new_wine = pd.read_csv('https://d32aokrjazspmn.cloudfront.net/materials/solvers_new_wine.csv')
new_wine

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,sulphates,alcohol
0,9.54,13.5,12.35,8.78,14.72,9.06,9.67,10.15,11.17,12.17


In [12]:
best_model = LogisticRegression(solver='lbfgs', max_iter=1000)
best_model.fit(X, y)

new_wine_scaled = scaler.transform(new_wine)

predicted_class = best_model.predict(new_wine_scaled)[0]
predicted_proba_of_class = best_model.predict_proba(new_wine_scaled)[0][predicted_class]

print("Tahmin edilen sınıf:", predicted_class)
print("Bu sınıfın olasılığı:", predicted_proba_of_class)

Tahmin edilen sınıf: 0
Bu sınıfın olasılığı: 0.9686233963189638


/home/kaan/.pyenv/versions/3.12.9/envs/workintech/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(
/home/kaan/.pyenv/versions/3.12.9/envs/workintech/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(


# 🏁  Kodunuzu kontrol edin ve notebook'unuzu gönderin

In [13]:
from nbresult import ChallengeResult

result = ChallengeResult(
    'new_data_prediction',
    predicted_class=predicted_class,
    predicted_proba_of_class=predicted_proba_of_class
)
result.write()
print(result.check())


============================= test session starts ==============================
platform linux -- Python 3.12.9, pytest-9.1.1, pluggy-1.6.0 -- /home/kaan/.pyenv/versions/3.12.9/envs/workintech/bin/python
cachedir: .pytest_cache
rootdir: /home/kaan/code/sprint_215/S16D4-S-data-solvers/tests
plugins: anyio-4.8.0, typeguard-4.4.2, dash-4.4.1
collecting ... collected 2 items

test_new_data_prediction.py::TestNewDataPrediction::test_predicted_class PASSED [ 50%]
test_new_data_prediction.py::TestNewDataPrediction::test_predicted_proba PASSED [100%]

============================== 2 passed in 0.22s ===============================


💯 You can commit your code:

git add tests/new_data_prediction.pickle

git commit -m 'Completed new_data_prediction step'

git push origin master

